# 02 — Data Inventory: `epl-00-25` (Premier League 2000/01 – 2024/25)

What does `raw_data/epl-00-25/epl_final.csv` contain? We look at its structure, columns, value ranges and coverage per season, and run some internal consistency checks.

It's **one row per match** from the home side's point of view, with full-time and half-time scores plus basic match stats (shots, corners, fouls, cards). The layout and short team names (`Man United`, `Nott'm Forest`) match [football-data.co.uk](https://www.football-data.co.uk/englandm.php), but the column names have been spelled out in full.

This notebook is **read-only**: it never modifies `raw_data/`.

In [1]:
from pathlib import Path

import pandas as pd

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 200)

# Resolve the repo root whether the notebook runs from notebooks/ or the root
ROOT = Path.cwd() if (Path.cwd() / "raw_data").exists() else Path.cwd().parent
DATA_DIR = ROOT / "raw_data" / "epl-00-25"
sorted(p.name for p in DATA_DIR.iterdir())

['epl_final.csv']

## 1. Load

In [2]:
df = pd.read_csv(DATA_DIR / "epl_final.csv", parse_dates=["MatchDate"])
print(f"shape: {df.shape}  |  file size: {(DATA_DIR / 'epl_final.csv').stat().st_size / 1024:.0f} KB")
df.head()

shape: (9380, 22)  |  file size: 715 KB


,Season,MatchDate,HomeTeam,AwayTeam,FullTimeHomeGoals,FullTimeAwayGoals,FullTimeResult,HalfTimeHomeGoals,HalfTimeAwayGoals,HalfTimeResult,HomeShots,AwayShots,HomeShotsOnTarget,AwayShotsOnTarget,HomeCorners,AwayCorners,HomeFouls,AwayFouls,HomeYellowCards,AwayYellowCards,HomeRedCards,AwayRedCards
0,2000/01,2000-08-19,Charlton,Man City,4,0,H,2,0,H,17,8,14,4,6,6,13,12,1,2,0,0
1,2000/01,2000-08-19,Chelsea,West Ham,4,2,H,1,0,H,17,12,10,5,7,7,19,14,1,2,0,0
2,2000/01,2000-08-19,Coventry,Middlesbrough,1,3,A,1,1,D,6,16,3,9,8,4,15,21,5,3,1,0
3,2000/01,2000-08-19,Derby,Southampton,2,2,D,1,2,A,6,13,4,6,5,8,11,13,1,1,0,0
4,2000/01,2000-08-19,Leeds,Everton,2,0,H,2,0,H,17,12,8,6,6,4,21,20,1,3,0,0


## 2. Column dictionary

Every stat comes as a `Home*` / `Away*` pair.

| Column | Meaning |
|---|---|
| `Season` | Season label, `YYYY/YY` (e.g. `2000/01`) |
| `MatchDate` | Match date |
| `HomeTeam`, `AwayTeam` | Teams (football-data.co.uk short names) |
| `FullTimeHomeGoals`, `FullTimeAwayGoals` | Full-time score |
| `FullTimeResult` | `H` home win / `D` draw / `A` away win |
| `HalfTimeHomeGoals`, `HalfTimeAwayGoals` | Half-time score |
| `HalfTimeResult` | `H` / `D` / `A` at half-time |
| `HomeShots`, `AwayShots` | Total shots |
| `HomeShotsOnTarget`, `AwayShotsOnTarget` | Shots on target |
| `HomeCorners`, `AwayCorners` | Corners won |
| `HomeFouls`, `AwayFouls` | Fouls committed |
| `HomeYellowCards`, `AwayYellowCards` | Yellow cards |
| `HomeRedCards`, `AwayRedCards` | Red cards |

There's no xG, possession, attendance, referee or matchweek column.

## 3. Types, missing values and ranges

In [3]:
def profile(df: pd.DataFrame) -> pd.DataFrame:
    return pd.DataFrame({
        "dtype": df.dtypes.astype(str),
        "non_null": df.notna().sum(),
        "null": df.isna().sum(),
        "null_pct": (df.isna().mean() * 100).round(1),
        "n_unique": df.nunique(),
        "min": df.min(),
        "max": df.max(),
    })

profile(df)

,dtype,non_null,null,null_pct,n_unique,min,max
Season,str,9380,0,0.0,25,2000/01,2024/25
MatchDate,datetime64[us],9380,0,0.0,2599,2000-08-19 00:00:00,2025-05-05 00:00:00
HomeTeam,str,9380,0,0.0,46,Arsenal,Wolves
AwayTeam,str,9380,0,0.0,46,Arsenal,Wolves
FullTimeHomeGoals,int64,9380,0,0.0,10,0,9
FullTimeAwayGoals,int64,9380,0,0.0,10,0,9
FullTimeResult,str,9380,0,0.0,3,A,H
HalfTimeHomeGoals,int64,9380,0,0.0,6,0,5
HalfTimeAwayGoals,int64,9380,0,0.0,6,0,5
HalfTimeResult,str,9380,0,0.0,3,A,H


In [4]:
df.describe().T.round(2)

,count,mean,min,25%,50%,75%,max,std
MatchDate,9380,2013-01-17 13:05:33.134328,2000-08-19 00:00:00,2006-12-05 18:00:00,2013-01-19 00:00:00,2019-03-02 00:00:00,2025-05-05 00:00:00,NaN
FullTimeHomeGoals,9380.0,1.535394,0.0,1.0,1.0,2.0,9.0,1.305432
FullTimeAwayGoals,9380.0,1.182623,0.0,0.0,1.0,2.0,9.0,1.157414
HalfTimeHomeGoals,9380.0,0.688273,0.0,0.0,0.0,1.0,5.0,0.835079
HalfTimeAwayGoals,9380.0,0.518763,0.0,0.0,0.0,1.0,5.0,0.735357
HomeShots,9380.0,13.617484,0.0,10.0,13.0,17.0,43.0,5.356424
AwayShots,9380.0,10.810661,0.0,7.0,10.0,14.0,37.0,4.696501
HomeShotsOnTarget,9380.0,5.973134,0.0,4.0,6.0,8.0,24.0,3.267954
AwayShotsOnTarget,9380.0,4.69371,0.0,3.0,4.0,6.0,20.0,2.750045
HomeCorners,9380.0,6.040299,0.0,4.0,6.0,8.0,20.0,3.110619


### Result distributions

In [5]:
pd.DataFrame({
    "full_time": df["FullTimeResult"].value_counts(normalize=True),
    "half_time": df["HalfTimeResult"].value_counts(normalize=True),
}).mul(100).round(1).rename_axis("result (%)")

,full_time,half_time
result (%),,
A,29.5,24.0
D,24.7,40.9
H,45.8,35.1


## 4. Coverage by season

A complete 20-team season has **380 matches**, which is **38 per team**.

In [6]:
teams_long = pd.concat([
    df[["Season", "HomeTeam"]].rename(columns={"HomeTeam": "team"}),
    df[["Season", "AwayTeam"]].rename(columns={"AwayTeam": "team"}),
])
per_team = teams_long.groupby(["Season", "team"]).size()

coverage = pd.DataFrame({
    "matches": df.groupby("Season").size(),
    "teams": df.groupby("Season")["HomeTeam"].nunique(),
    "first": df.groupby("Season")["MatchDate"].min().dt.date,
    "last": df.groupby("Season")["MatchDate"].max().dt.date,
    "min_per_team": per_team.groupby("Season").min(),
    "max_per_team": per_team.groupby("Season").max(),
})
coverage["complete"] = coverage["matches"].eq(380)
coverage

,matches,teams,first,last,min_per_team,max_per_team,complete
Season,,,,,,,
2000/01,380,20,2000-08-19,2001-05-19,38,38,True
2001/02,380,20,2001-08-18,2002-05-11,38,38,True
2002/03,380,20,2002-08-17,2003-05-11,38,38,True
2003/04,335,20,2003-08-16,2004-05-08,33,35,False
2004/05,335,20,2004-08-14,2005-04-20,32,34,False
2005/06,380,20,2005-08-13,2006-05-07,38,38,True
2006/07,380,20,2006-08-19,2007-05-13,38,38,True
2007/08,380,20,2007-08-11,2008-05-11,38,38,True
2008/09,380,20,2008-08-16,2009-05-24,38,38,True


In [7]:
incomplete = coverage[~coverage["complete"]]
print(f"incomplete seasons: {incomplete.index.tolist()}")
print(f"missing matches:    {(380 - incomplete['matches']).to_dict()}")

incomplete seasons: ['2003/04', '2004/05', '2024/25']
missing matches:    {'2003/04': 45, '2004/05': 45, '2024/25': 30}


The incomplete seasons:
- **2003/04 and 2004/05** each have 335 of 380 matches. The gaps are spread across the season (every team is 1–6 matches short), so rows were dropped at random rather than the season being cut off. `epl-92-23` has these 90 matches (see notebook 03).
- **2024/25** stops on 2025-05-05 at 35 matches per team. The file was saved mid-season, so the last three matchweeks (30 matches) are missing. They're in `raw_data/2425.csv` (see notebook 06).

## 5. Teams

In [8]:
seasons_per_team = teams_long.drop_duplicates().groupby("team").size().sort_values(ascending=False)
print(f"{len(seasons_per_team)} distinct teams")
seasons_per_team.rename("seasons_in_file").to_frame().T

46 distinct teams


team,Arsenal,Chelsea,Tottenham,Man United,Liverpool,Everton,Man City,Newcastle,Aston Villa,West Ham,Fulham,Southampton,Sunderland,Crystal Palace,Leicester,West Brom,Blackburn,Bolton,Wolves,Middlesbrough,Stoke,Burnley,Wigan,Bournemouth,Brighton,Portsmouth,Birmingham,Charlton,Watford,Norwich,Leeds,Swansea,Hull,Brentford,Sheffield United,Ipswich,Nott'm Forest,Reading,QPR,Derby,Cardiff,Huddersfield,Bradford,Blackpool,Luton,Coventry
seasons_in_file,25,25,25,25,25,25,24,23,22,22,18,17,14,13,13,13,11,11,11,10,10,9,8,8,8,7,7,7,7,7,7,7,5,4,4,3,3,3,3,3,2,2,1,1,1,1


## 6. Internal consistency checks

In [9]:
def result(h, a):
    return pd.Series(["H" if x > y else "A" if x < y else "D" for x, y in zip(h, a)], index=h.index)

checks = {
    "duplicate rows": df.duplicated().sum(),
    "duplicate fixtures (season, home, away)": df.duplicated(["Season", "HomeTeam", "AwayTeam"]).sum(),
    "team plays itself": (df["HomeTeam"] == df["AwayTeam"]).sum(),
    "FT result != FT score": (result(df["FullTimeHomeGoals"], df["FullTimeAwayGoals"]) != df["FullTimeResult"]).sum(),
    "HT result != HT score": (result(df["HalfTimeHomeGoals"], df["HalfTimeAwayGoals"]) != df["HalfTimeResult"]).sum(),
    "HT goals > FT goals": ((df["HalfTimeHomeGoals"] > df["FullTimeHomeGoals"]) | (df["HalfTimeAwayGoals"] > df["FullTimeAwayGoals"])).sum(),
    "shots on target > shots": ((df["HomeShotsOnTarget"] > df["HomeShots"]) | (df["AwayShotsOnTarget"] > df["AwayShots"])).sum(),
    "goals > shots on target": ((df["FullTimeHomeGoals"] > df["HomeShotsOnTarget"]) | (df["FullTimeAwayGoals"] > df["AwayShotsOnTarget"])).sum(),
    "date outside season window": (~(df["MatchDate"].dt.year - df["Season"].str[:4].astype(int)).isin([0, 1])).sum(),
}
pd.Series(checks, name="violations").to_frame()

,violations
duplicate rows,0
"duplicate fixtures (season, home, away)",0
team plays itself,0
FT result != FT score,0
HT result != HT score,0
HT goals > FT goals,0
shots on target > shots,3
goals > shots on target,47
date outside season window,0


Rows where on-target shots are more than total shots are data-entry errors. Goals can outnumber shots on target legitimately, because own goals don't count as the scoring team's shots. So that check is for information only.

In [10]:
df[(df["HomeShotsOnTarget"] > df["HomeShots"]) | (df["AwayShotsOnTarget"] > df["AwayShots"])][
    ["Season", "MatchDate", "HomeTeam", "AwayTeam", "HomeShots", "HomeShotsOnTarget", "AwayShots", "AwayShotsOnTarget"]
]

,Season,MatchDate,HomeTeam,AwayTeam,HomeShots,HomeShotsOnTarget,AwayShots,AwayShotsOnTarget
80,2000/01,2000-10-14,Coventry,Tottenham,0,5,4,5
317,2000/01,2001-04-13,Bradford,Charlton,4,8,3,8
7898,2021/22,2021-08-15,Newcastle,West Ham,17,3,8,9


## 7. Summary

| | |
|---|---|
| **Grain** | One row per match (home perspective), 9,380 rows × 22 columns |
| **Span** | 2000/01 – 2024/25 (25 seasons), dates 2000-08-19 to 2025-05-05 |
| **Nulls** | None |
| **Stats** | FT/HT score and result, shots, shots on target, corners, fouls, yellow/red cards |

**Things to keep in mind downstream**
- **Three seasons are incomplete:** 2003/04 and 2004/05 (335/380 each, gaps scattered) and 2024/25 (350/380, last three matchweeks missing). Season tables built from this file alone will be wrong for these seasons. The missing 2003–05 matches are in `epl-92-23` and the missing 2024/25 matches are in `raw_data/2425.csv`.
- **3 rows have shots on target > total shots.** These are data-entry errors; fix or exclude them before shot-based analysis.
- **Team names are short-form** (`Man United`, `Nott'm Forest`, `Leeds`). They differ from both `epl-92-23` (`Manchester Utd`, `Nott'ham Forest`) and `epl-25-26` (FBref names), so joining across datasets needs a name crosswalk (started in notebook 03).
- There's no matchweek column. Derive one from date order if you need it.